# Masked Self-Attention: Concepts and Code in PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

---
<a id='1'></a>
# Part 1: Causal (Masked) Self-Attention

Same attention as full self-attention, but position **i** may only attend to positions **j ≤ i** — not future tokens. Used in **decoder-only** models (GPT, LLaMA).

Let **M** be a mask with $M_{ij} = -\infty$ when $j > i$, else $0$. Then:

$$\text{MaskedAttention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

In code we add `-inf` on the **upper triangle** of the score matrix (strictly above the diagonal) before softmax, so those weights become 0.

- **Q, K, V:** same roles as in standard self-attention
- **√d_k:** same scaling as in "Attention Is All You Need"

In [ ]:
class MaskedSelfAttention(nn.Module):

    def __init__(self, d_model=2, row_dim=0, col_dim=1):
        ## d_model = number of embedding values per token
        ## default d_model=2 so we can verify math by hand
        super().__init__()

        ## W_q, W_k, W_v: learned projection matrices, no bias per original paper
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

        self.row_dim = row_dim
        self.col_dim = col_dim

    def forward(self, token_encodings):
        ## Step 1: Project inputs into Q, K, V
        q = self.W_q(token_encodings)
        k = self.W_k(token_encodings)
        v = self.W_v(token_encodings)

        seq_len = token_encodings.size(self.row_dim)

        ## Step 2: Similarity scores Q @ K^T
        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))

        ## Step 3: Scale by sqrt(d_k)
        scaled_sims = sims / torch.tensor(k.size(self.col_dim) ** 0.5)

        ## Step 4: Causal mask — cannot attend to future positions (j > i)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=scaled_sims.device, dtype=torch.bool),
            diagonal=1,
        )
        scaled_sims = scaled_sims.masked_fill(causal_mask, float("-inf"))

        ## Step 5: Softmax -> attention weights
        attention_percents = F.softmax(scaled_sims, dim=self.col_dim)

        ## Step 6: Weighted sum of values
        attention_scores = torch.matmul(attention_percents, v)

        return attention_scores

In [ ]:
## 3 tokens, each with a 2-dim embedding (same toy setup as the self-attention notebook)
encodings_matrix = torch.tensor([[1.16, 0.23],
                                  [0.57, 1.36],
                                  [4.41, -2.16]])

torch.manual_seed(42)

maskedAttention = MaskedSelfAttention(d_model=2, row_dim=0, col_dim=1)

maskedAttention(encodings_matrix)

In [ ]:
print("W_q:\n", maskedAttention.W_q.weight.transpose(0, 1))
print("\nW_k:\n", maskedAttention.W_k.weight.transpose(0, 1))
print("\nW_v:\n", maskedAttention.W_v.weight.transpose(0, 1))

In [ ]:
q = maskedAttention.W_q(encodings_matrix)
k = maskedAttention.W_k(encodings_matrix)
v = maskedAttention.W_v(encodings_matrix)

print("Q:\n", q)
print("\nK:\n", k)
print("\nV:\n", v)

In [ ]:
sims = torch.matmul(q, k.transpose(dim0=0, dim1=1))
print("Similarity scores (Q @ K^T):\n", sims)

In [ ]:
scaled_sims = sims / (torch.tensor(2) ** 0.5)
print("Scaled similarities:\n", scaled_sims)

In [ ]:
seq_len = encodings_matrix.size(0)
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
print("Causal mask (True = masked future position):\n", causal_mask)
scaled_masked = scaled_sims.masked_fill(causal_mask, float("-inf"))
print("\nScaled scores after mask (future = -inf):\n", scaled_masked)

In [ ]:
attention_percents = F.softmax(scaled_masked, dim=1)
print("Attention weights:\n", attention_percents)
print("\nRow sums (should all be 1.0):", attention_percents.sum(dim=1))
print("\nFuture weights (strictly upper triangle) should be 0:")
print(torch.triu(attention_percents, diagonal=1))

In [ ]:
output = torch.matmul(attention_percents, maskedAttention.W_v(encodings_matrix))
print("Manual output:\n", output)
print("\nMatches forward pass?", torch.allclose(output, maskedAttention(encodings_matrix), atol=1e-6))